# Medium Article Semantic Search by Title+Subtitle

### Load Data

In [16]:
import pandas as pd

In [17]:
df = pd.read_csv("medium_post_titles.csv", nrows=10000) # excercise whole data set
# data source: https://www.kaggle.com/datasets/nulldata/medium-post-titles

In [18]:
df.head()

,category,title,subtitle,subtitle_truncated_flag
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False
3,equality,"""Call me Nat Love"" :Black Cowboys and the Fron...",NaN,False
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False


In [19]:
# df["subtitle_truncated_flag"].value_counts()
df["subtitle_truncated_flag"].value_counts()

subtitle_truncated_flag
False    6318
True     3682
Name: count, dtype: int64

### Data Cleanup

In [20]:
# # df.isna().sum()

# df = df.dropna()
# df = df[~df["subtitle_truncated_flag"]]
# # df["subtitle_truncated_flag"].value_counts()

# df['title_extended'] = df['title'] + df['subtitle']

df.isna().sum()


category                     0
title                        0
subtitle                   107
subtitle_truncated_flag      0
dtype: int64

In [21]:
df = df.dropna()

In [22]:
df1 = df[~df["subtitle_truncated_flag"]]
# print(df1)

In [23]:
df["subtitle_truncated_flag"].value_counts()

subtitle_truncated_flag
False    6211
True     3682
Name: count, dtype: int64

In [24]:
df.shape

(9893, 4)

In [25]:
df["title_extended"] = df["title"]+df["subtitle"]

In [26]:
df.head()

,category,title,subtitle,subtitle_truncated_flag,title_extended
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False,"""21 Conversations"" - A fun (and easy) game for..."
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False,"""Biblical Porn"" at Mars HillAuthor and UW lect..."
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False,"""CISGENDER?! Is That A Disease?!""Or, a primer ..."
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we..."
5,cryptocurrency,"""Cypherpunks and Wall Street"": The Security To...",Bruce Fenton presents at the World Blockchain ...,False,"""Cypherpunks and Wall Street"": The Security To..."


In [27]:
# df.head()
# df['category'].nunique()  # metadata
# df.shape # 6k vectors, full set in excercise

df["category"].nunique()



93

In [28]:
df.shape

(9893, 5)

In [29]:
df.head()

,category,title,subtitle,subtitle_truncated_flag,title_extended
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False,"""21 Conversations"" - A fun (and easy) game for..."
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False,"""Biblical Porn"" at Mars HillAuthor and UW lect..."
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False,"""CISGENDER?! Is That A Disease?!""Or, a primer ..."
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we..."
5,cryptocurrency,"""Cypherpunks and Wall Street"": The Security To...",Bruce Fenton presents at the World Blockchain ...,False,"""Cypherpunks and Wall Street"": The Security To..."


### Prep for Upsert

In [30]:
# API_KEY =""


In [31]:
# init pinecone
from pinecone import Pinecone, ServerlessSpec
# API_KEY = "YOUR API KEY"
pc = Pinecone(api_key = API_KEY)

In [75]:
# pc.create_index(name = "medium-data", 
#                 dimension=384, 
#                 metric="cosine",
#                 spec=ServerlessSpec(
#                     cloud="aws",
#                     region="us-east-1"
#                 )) # remember to use only us-east-1 in free tier


pc.create_index(name="medium-dates",
               dimension=384,
               metric="cosine",
               spec=ServerlessSpec(
                   cloud="aws",
                   region="us-east-1"
               ))

{
    "name": "medium-dates",
    "metric": "cosine",
    "host": "medium-dates-kbs8m4f.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "20

In [34]:
pc.list_indexes()

[
    {
        "name": "hello-pinecone",
        "metric": "cosine",
        "host": "hello-pinecone-kbs8m4f.svc.apw5-4e34-81fa.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-west-2",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 4,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "test",
        "metric": "cosine",
        "host": "test-kbs8m4f.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
         

In [35]:
# pinecone.create_index(name='medium-data', dimension=384, pod_type='s1', metric="cosine" )

In [36]:
# !pip install sentence-transformers

In [37]:
# !python -m pip install --upgrade pip

In [38]:
# !pip install tf-keras


In [39]:
# !pip install --upgrade transformers sentence-transformers tensorflow tf-keras

In [40]:
from sentence_transformers import SentenceTransformer
import torch

In [48]:
# !pip install ipywidgets


In [50]:
from sentence_transformers import SentenceTransformer

print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print("Model loaded successfully")

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [45]:
model = SentenceTransformer('all-MiniLM-L6-v2')# cuda or cpu
# model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [42]:
# !python -m pip install torch torchvision torchaudio

In [43]:
# !pip uninstall torch torchvision torchaudio -y
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [44]:
df.head()

,category,title,subtitle,subtitle_truncated_flag,title_extended
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False,"""21 Conversations"" - A fun (and easy) game for..."
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False,"""Biblical Porn"" at Mars HillAuthor and UW lect..."
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False,"""CISGENDER?! Is That A Disease?!""Or, a primer ..."
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we..."
5,cryptocurrency,"""Cypherpunks and Wall Street"": The Security To...",Bruce Fenton presents at the World Blockchain ...,False,"""Cypherpunks and Wall Street"": The Security To..."


In [52]:
# df['values'] = df['title_extended'].map(
#     lambda x: (model.encode(x)).tolist()) # python list, 6k rows 1 min

df['values'] = df['title_extended'].map(
    lambda x: (model.encode(x)).tolist())


In [54]:
df.head()

,category,title,subtitle,subtitle_truncated_flag,title_extended,values
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False,"""21 Conversations"" - A fun (and easy) game for...","[-0.031074360013008118, -0.014303414151072502,..."
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False,"""Biblical Porn"" at Mars HillAuthor and UW lect...","[-0.03467034548521042, -0.018165316432714462, ..."
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False,"""CISGENDER?! Is That A Disease?!""Or, a primer ...","[0.03740720450878143, -0.0008568223565816879, ..."
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we...","[-0.013686500489711761, 0.004296017810702324, ..."
5,cryptocurrency,"""Cypherpunks and Wall Street"": The Security To...",Bruce Fenton presents at the World Blockchain ...,False,"""Cypherpunks and Wall Street"": The Security To...","[-0.031468793749809265, -0.004646690096706152,..."


In [55]:
# df['id'] = df.reset_index(drop = 'index').index

df['id'] = df.reset_index(drop='index').index


In [57]:
# df.head()

In [58]:
# df['metadata'] = df.apply(lambda x: {
#     'title' : x['title'],
#     'subtitle': x['subtitle'],
#     'category': x['category']
    
# }, axis=1)

df['metadata'] = df.apply(lambda x: {
    'title': x['title'],
    'subtitle':x['subtitle'],
    'category':x['category']
}, axis=1)

In [59]:
# df_upsert = df[['id', 'values', 'metadata']]
df_upsert = df[['id','values','metadata']]

In [62]:
# df_upsert['id'] = df_upsert['id'].map(lambda x: str(x))
# df_upsert['id'] = df_upsert['id'].map(lambda x: str(x))

In [70]:
print(pc.list_indexes().names())
print(pc)
from pinecone import Pinecone

# pc = Pinecone(api_key="")
print(pc.list_indexes().names())

['hello-pinecone', 'test', 'medium-date', 'insert']
['hello-pinecone', 'test', 'medium-date', 'insert']


In [72]:
import pinecone
print(pinecone.__version__)

8.1.2


In [84]:
from pinecone import ServerlessSpec

pc.create_index(
    name="test2",
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "test2",
    "metric": "cosine",
    "host": "test2-kbs8m4f.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
      

In [85]:
from pinecone import Pinecone
# pc=Pinecone(api_key="")

In [86]:
index =pc.Index("test2")

In [87]:
index.upsert_from_dataframe(df_upsert) # 6k takes 1 min

sending upsert requests:   0%|          | 0/9893 [00:00<?, ?it/s]

UpsertResponse(upserted_count=9893, _response_info={'raw_headers': {'date': 'Mon, 27 Apr 2026 10:25:11 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '20', 'x-pinecone-request-logical-size': '679810', 'x-pinecone-request-latency-ms': '2211', 'x-envoy-upstream-service-time': '224', 'x-pinecone-response-duration-ms': '2213', 'grpc-status': '0', 'server': 'envoy'}})

### Query

In [92]:
xc = index.query(vector=(model.encode("which city is the most beautiful")).tolist(), # python list
           top_k=10,
           include_metadata=True)

# xc = index.query(vector=(model.encode("which city is the most beautifull")).tolist(),
#                 top_k=10,
#                 include_metadata=True)

In [94]:
# for result in xc['matches']:
#     print(f"{round(result['score'], 2)}: {result['metadata']['title']}: {result['metadata']['category']} ")

for result in xc['matches']:
    print(f"{round(result['score'],2)}: {result['metadata']['title']}: {result['metadata']['category']}")

0.57: 3 Places Where You Can Find Beauty: photography
0.46: 6 Easy Reasons to Enjoy Exploring South Wales: travel
0.45: A City That’s Better for the Blind Is Better for Everyone: accessibility
0.45: A Shining City on a Hill: politics
0.44: A Just City is a Healthy City: environment
0.42: A Most Beautiful Game: sports
0.41: A Case For Preserving History: education
0.4: 6 Literary Cities for Book Lovers To Visit This Year: travel
0.4: Ace Hotel: A UX Case Study: ux
0.39: A city and its architecture: cities


In [95]:
for result in xc['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['subtitle']}: {result['metadata']['category']} ")

0.57: If you are willing to look hard enough, eventually you will see beauty in the most difficult of places.: photography 
0.46: Pembrokeshire is as beautiful as the Italian Coast.: travel 
0.45: Complete parity with the sighted may seem like an impossible goal, but maybe the only thing holding us back is a lack of imagination.: accessibility 
0.45: What does America stand for?: politics 
0.44: At CultureHouse, we believe that urban design can be used for the common good, and that good design should be an essential part of the: environment 
0.42: The World Cup gets advertising right: sports 
0.41: I am fortunate to live in a town that is both beautiful and historic. When I say historic, I mean, of course, from an American perspective: education 
0.4: Combine your love for books and travel with these 6 literary cities.: travel 
0.4: Discover the city you are visting like a local: ux 
0.39: Bangalore Chapter: cities 


### Excercise: Upsert all data

In [96]:
df.head()

,category,title,subtitle,subtitle_truncated_flag,title_extended,values,id,metadata
0,work,"""21 Conversations"" - A fun (and easy) game for...",A (new?) Icebreaker game to get your team to s...,False,"""21 Conversations"" - A fun (and easy) game for...","[-0.031074360013008118, -0.014303414151072502,...",0,"{'title': '""21 Conversations"" - A fun (and eas..."
1,spirituality,"""Biblical Porn"" at Mars Hill",Author and UW lecturer Jessica Johnson talks a...,False,"""Biblical Porn"" at Mars HillAuthor and UW lect...","[-0.03467034548521042, -0.018165316432714462, ...",1,"{'title': '""Biblical Porn"" at Mars Hill', 'sub..."
2,lgbtqia,"""CISGENDER?! Is That A Disease?!""","Or, a primer in gender vocabulary for the curi...",False,"""CISGENDER?! Is That A Disease?!""Or, a primer ...","[0.03740720450878143, -0.0008568223565816879, ...",2,"{'title': '""CISGENDER?! Is That A Disease?!""',..."
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we...","[-0.013686500489711761, 0.004296017810702324, ...",3,"{'title': '""Can I Train my Model on Your Compu..."
5,cryptocurrency,"""Cypherpunks and Wall Street"": The Security To...",Bruce Fenton presents at the World Blockchain ...,False,"""Cypherpunks and Wall Street"": The Security To...","[-0.031468793749809265, -0.004646690096706152,...",4,"{'title': '""Cypherpunks and Wall Street"": The ..."
